# Tarea para el Hogar 04

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Oct 22 23:57:07 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671199,35.9,1479650,79.1,1479650,79.1
Vcells,1242607,9.5,8388608,64.0,1978693,15.1


### 2.2.2 Carga de Librerias

In [3]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

# Optuna se usa desde R: LightGBM y los datos permanecen en R.
# Ejecutar al iniciar la sesión R, antes de importar módulos de Python.
if (!requireNamespace("reticulate", quietly = TRUE) ||
    utils::packageVersion("reticulate") < "1.41.0") {
  install.packages("reticulate")
}
Sys.setenv(RETICULATE_PYTHON = "managed")
reticulate::py_require("optuna>=4.0,<6.0")
optuna <- reticulate::import("optuna", convert = TRUE)

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: parallel

Loading required package: primes

Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm



### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [4]:
PARAM <- list()
# Carpeta nueva para comparar con las versiones que optimizan AUC.
PARAM$experimento <- "5949"
PARAM$semilla_primigenia <- 149911

In [5]:
PARAM$kaggle$competencia <- "labo-1-virtual-inicial"
PARAM$kaggle$cortes <- seq(9000L, 14000L, by = 500L)

# Ganancia NETA por cliente contactado, según las reglas de la competencia.
# El cliente no contactado aporta cero.
PARAM$ganancia$baja2 <- 975000
PARAM$ganancia$otros <- -25000
# Corte de referencia fijado antes de la búsqueda; no maximizo sobre los cortes.
# Se evalúa la misma proporción de contactos en cada fold completo de julio.
PARAM$ganancia$envios_objetivo <- 12500L

In [6]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [7]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# Presupuesto para cinco dimensiones: 15 ensayos iniciales y luego TPE.
PARAM$hyperparametertuning$n_trials <- 80L
PARAM$hyperparametertuning$n_startup_trials <- 15L

# Cantidad fija en cada ensayo y en el entrenamiento final.
PARAM$hyperparametertuning$num_iterations <- 1000L

# Espacio inicial para aprendizaje, complejidad y regularización.
# Los rangos son un punto de partida, no una importancia medida en estos datos.
PARAM$hyperparametertuning$espacio <- list(
  learning_rate = c(0.01, 0.15),
  num_leaves = c(16L, 128L),
  min_data_in_leaf = c(20L, 2000L),
  feature_fraction = c(0.5, 1.0),
  bagging_fraction = c(0.6, 1.0)
)

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # fijo en cero; no participa de Optuna
  lambda_l2= 0.0, # fijo en cero; no participa de Optuna
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_freq= 1L, # activa el bagging_fraction propuesto por Optuna
  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= PARAM$hyperparametertuning$num_iterations,
  learning_rate= 0.10,  # >=0
  feature_fraction= 1.0, # 0 < ff <= 1.0
  num_leaves= 32, # integer >= 2
  min_data_in_leaf= 20 # integer >= 0
)

### 2.2.4  Preprocesamiento

In [8]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [9]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [10]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [11]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [12]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [13]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [14]:
# CV sobre TODO julio; el undersampling se aplica solo al entrenamiento.
# "training" conserva la máscara reproducible construida en la celda anterior.
n_folds <- as.integer(PARAM$hyperparametertuning$xval_folds)
stopifnot(n_folds >= 2L, !anyNA(dataset_train$clase_ternaria),
          all(dataset_train$clase_ternaria %in% c("BAJA+2", "BAJA+1", "CONTINUA")),
          !anyNA(dataset_train$numero_de_cliente),
          !anyDuplicated(dataset_train$numero_de_cliente))

# Obtengo la cantidad real de clientes futuros: la descripción publicada
# contiene dos cantidades de filas distintas y no conviene hardcodearlas.
ids_futuro <- dataset[foto_mes == 202109, numero_de_cliente]
n_futuro <- length(ids_futuro)
stopifnot(n_futuro > 0L, !anyNA(ids_futuro), !anyDuplicated(ids_futuro),
          all(PARAM$kaggle$cortes > 0 & PARAM$kaggle$cortes <= n_futuro),
          PARAM$ganancia$envios_objetivo > 0L,
          PARAM$ganancia$envios_objetivo <= n_futuro)
PARAM$ganancia$n_clientes_futuro <- n_futuro
PARAM$ganancia$proporcion_objetivo <- PARAM$ganancia$envios_objetivo / n_futuro
PARAM$ganancia$unidad_cv <- "millones proyectados a toda la poblacion futura"

# Estratifico por las tres clases para que BAJA+2 esté representada en cada fold.
# fold_id queda fuera de la tabla: nunca se incorpora como predictor.
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
fold_id <- integer(nrow(dataset_train))
for (clase in unique(dataset_train$clase_ternaria)) {
  indices <- which(dataset_train$clase_ternaria == clase)
  if (length(indices) < n_folds) stop("Faltan casos de una clase para los folds.")
  fold_id[indices] <- sample(rep(seq_len(n_folds), length.out = length(indices)))
}

x_julio <- data.matrix(dataset_train[, campos_buenos, with = FALSE])
cv_cache <- vector("list", n_folds)
for (fold in seq_len(n_folds)) {
  idx_valid <- which(fold_id == fold)
  idx_train <- which(fold_id != fold & dataset_train$training == 1L)
  stopifnot(length(idx_train) > 0L, length(idx_valid) > 0L,
            !any(idx_train %in% idx_valid))
  dtrain_fold <- lgb.Dataset(
    data = x_julio[idx_train, , drop = FALSE],
    label = dataset_train$clase01[idx_train],
    params = list(max_bin = PARAM$lgbm$param_fijos$max_bin,
                  feature_pre_filter = FALSE),
    free_raw_data = TRUE
  )
  # Construyo y libero la copia cruda de cada entrenamiento una sola vez.
  lgb.Dataset.construct(dtrain_fold)
  cv_cache[[fold]] <- list(dtrain = dtrain_fold,
                          idx_train = idx_train, idx_valid = idx_valid)
}
data.table(fold = seq_len(n_folds),
           entrenamiento = vapply(cv_cache, function(z) length(z$idx_train), integer(1)),
           validacion_completa = vapply(cv_cache, function(z) length(z$idx_valid), integer(1)))

fold,entrenamiento,validacion_completa
<int>,<int>,<int>
1,66937,32897
2,66950,32897
3,66928,32896
4,66875,32895
5,66970,32894


### 2.2.5 Optuna por ganancia, con 1000 iteraciones fijas

La función objetivo es la **ganancia media de validación cruzada** para una
proporción de contactos equivalente a **12.500 clientes de septiembre**.
Este corte se fija en `PARAM$ganancia$envios_objetivo` antes de la búsqueda.
Se informa también la curva de 9.000 a 14.000, sin usar su máximo para elegir
cada ensayo. Los cinco hiperparámetros y sus rangos siguen en la configuración.

* Cada `BAJA+2` contactado suma $975.000. Cada `BAJA+1` o `CONTINUA`
  contactado resta $25.000. Los no contactados aportan cero.
* Se conserva el objetivo binario original de entrenamiento (`BAJA+1` o
  `BAJA+2` frente a `CONTINUA`). **Solo `BAJA+2` recibe la recompensa económica**.
  La AUC de esa etiqueta binaria se registra como diagnóstico secundario;
  Optuna recibe la ganancia, no la AUC.
* Los mismos 5 folds estratificados se reutilizan en todos los ensayos.
  El undersampling 0.5 se aplica **solo a sus entrenamientos**; la validación
  conserva todos los clientes y la proporción natural de las tres clases.
* Cada fold contacta `round(n_validacion * 12500 / n_clientes_septiembre)`
  clientes de mayor score. Su ganancia se divide por el tamaño del fold,
  se multiplica por la población de septiembre y se expresa en millones.
  Optuna maximiza la media de esas proyecciones. El desvío entre folds se
  informa para mostrar variabilidad. Esto **no es una predicción de la ganancia
  pública**: el público contiene solo 30% y septiembre puede diferir de julio.
* Las **1000 rondas son fijas**, sin early stopping ni pruning. Ambos lambdas
  permanecen en cero. El primer ensayo evalúa los hiperparámetros del 5942
  con este nuevo protocolo; los 80 ensayos totales incluyen esa referencia.
  Los primeros 15 ensayos completos (referencia y propuestas aleatorias)
  constituyen el arranque antes de la adaptación de TPE.
* Solo julio tiene etiquetas: esta CV permite comparar candidatos localmente,
  pero sus resultados se reutilizan para optimizar y no son una prueba
  independiente del ganador ni una validación temporal de septiembre.

La condición económica teórica `P(BAJA+2) > 0.025` exige probabilidades
calibradas para **esa clase y la población real**. No se aplica directamente
al score de este modelo, entrenado con otra etiqueta y undersampling.

Se guardan `tb_optuna_ganancia_1000.txt` y `tb_curvas_ganancia_1000.txt`.
La producción conserva el entrenamiento completo y el ajuste de
`min_data_in_leaf` del notebook original. Los cortes de envío se mantienen.
El leaderboard público sirve como contraste externo; elegir repetidamente
modelos y cortes por su resultado puede sobreajustar ese 30%.

In [15]:
# AUC de la etiqueta binaria original; no es el objetivo de Optuna.
Calcular_AUC <- function(y, prob) {
  n1 <- sum(y == 1L)
  n0 <- sum(y == 0L)
  if (n1 == 0L || n0 == 0L) stop("Un fold no contiene ambas clases binarias.")
  (sum(rank(prob, ties.method = "average")[y == 1L]) - n1 * (n1 + 1) / 2) /
    (n1 * n0)
}

Ganancias_por_corte <- function(clase, prob, cortes, n_poblacion) {
  stopifnot(length(clase) == length(prob), length(prob) > 0L,
            all(is.finite(prob)), !anyNA(clase),
            all(clase %in% c("BAJA+2", "BAJA+1", "CONTINUA")),
            all(cortes >= 0 & cortes <= n_poblacion), n_poblacion > 0L)
  # El índice original resuelve los empates de manera reproducible.
  orden <- order(-prob, seq_along(prob))
  aportes <- ifelse(clase[orden] == "BAJA+2",
                    PARAM$ganancia$baja2, PARAM$ganancia$otros)
  acumulada <- c(0, cumsum(aportes))
  k <- as.integer(round(length(prob) * cortes / n_poblacion))
  # El primer elemento es ganancia cero para k = 0.
  ganancia_fold <- acumulada[k + 1L]
  data.table(corte = as.integer(cortes), contactos_fold = k,
             ganancia_fold = ganancia_fold,
             Ganancia_cv_proyectada_millones =
               ganancia_fold / length(prob) * n_poblacion / 1e6)
}

Estimar_Ganancia_lightgbm <- function(x) {
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
  rondas <- as.integer(PARAM$hyperparametertuning$num_iterations)
  param_completo$num_iterations <- rondas
  cortes <- sort(unique(c(PARAM$kaggle$cortes, PARAM$ganancia$envios_objetivo)))
  curvas_folds <- vector("list", length(cv_cache))
  auc_folds <- numeric(length(cv_cache))

  for (fold in seq_along(cv_cache)) {
    cache <- cv_cache[[fold]]
    # La semilla de LightGBM y los datos de cada fold son los mismos en
    # todos los ensayos; solo cambian los hiperparámetros propuestos.
    modelo_fold <- lgb.train(params = param_completo, data = cache$dtrain,
                             nrounds = rondas)
    prob <- predict(modelo_fold, x_julio[cache$idx_valid, , drop = FALSE],
                    num_iteration = rondas)
    auc_folds[fold] <- Calcular_AUC(dataset_train$clase01[cache$idx_valid], prob)
    curva <- Ganancias_por_corte(
      dataset_train$clase_ternaria[cache$idx_valid], prob, cortes, n_futuro
    )
    curva[, fold := fold]
    curvas_folds[[fold]] <- curva
    rm(modelo_fold)
  }
  gc(full = TRUE, verbose = FALSE)
  curvas_folds <- rbindlist(curvas_folds)
  curva_media <- curvas_folds[, .(
    Ganancia_cv_proyectada_millones = mean(Ganancia_cv_proyectada_millones),
    Ganancia_sd = sd(Ganancia_cv_proyectada_millones)
  ), by = corte]
  objetivo <- curva_media[corte == PARAM$ganancia$envios_objetivo]
  stopifnot(nrow(objetivo) == 1L,
            is.finite(objetivo$Ganancia_cv_proyectada_millones),
            all(is.finite(auc_folds)))
  list(Ganancia = objetivo$Ganancia_cv_proyectada_millones,
       Ganancia_sd = objetivo$Ganancia_sd, AUC = mean(auc_folds),
       num_iterations = rondas, curva = curva_media)
}

In [16]:
# Estudio nuevo: nunca se mezclan objetivos de AUC y ganancia.
stopifnot(PARAM$hyperparametertuning$n_trials > 1L,
          PARAM$hyperparametertuning$n_startup_trials > 0L,
          PARAM$hyperparametertuning$n_startup_trials < PARAM$hyperparametertuning$n_trials,
          PARAM$hyperparametertuning$num_iterations == 1000L)

espacio <- copy(PARAM$hyperparametertuning$espacio)
n_train_cv <- min(vapply(cv_cache, function(z) length(z$idx_train), integer(1)))
limite_hoja <- max(2L, as.integer(floor(n_train_cv / 4)))
espacio$min_data_in_leaf[2] <- min(espacio$min_data_in_leaf[2], limite_hoja)
espacio$min_data_in_leaf[1] <- min(espacio$min_data_in_leaf)
PARAM$hyperparametertuning$espacio_efectivo <- copy(espacio)

sampler <- optuna$samplers$TPESampler(
  seed = as.integer(PARAM$semilla_primigenia),
  n_startup_trials = as.integer(PARAM$hyperparametertuning$n_startup_trials),
  multivariate = FALSE
)
estado_fallido <- reticulate::import("optuna.trial", convert = FALSE)$TrialState$FAIL
estudio <- optuna$create_study(
  direction = "maximize", sampler = sampler,
  study_name = paste0("HT", PARAM$experimento, "_ganancia_1000")
)

# Referencia 5942 con el mismo protocolo de CV que todos los candidatos.
# No reutilizo su AUC o su score de Kaggle como observación del estudio.
referencia_5942 <- list(learning_rate = 0.01, num_leaves = 64L,
                       min_data_in_leaf = 500L,
                       feature_fraction = 0.75, bagging_fraction = 1.0)
for (nombre in names(referencia_5942)) {
  if (referencia_5942[[nombre]] < espacio[[nombre]][1] ||
      referencia_5942[[nombre]] > espacio[[nombre]][2]) {
    stop("La referencia 5942 queda fuera del espacio efectivo: ", nombre)
  }
}
estudio$enqueue_trial(params = referencia_5942)

tb_nueva <- data.table(
  ensayo = integer(), learning_rate = numeric(), num_leaves = integer(),
  min_data_in_leaf = integer(), feature_fraction = numeric(),
  bagging_fraction = numeric(), num_iterations = integer(),
  Ganancia_cv_proyectada_millones = numeric(), Ganancia_sd = numeric(), AUC = numeric()
)
tb_curvas <- data.table(ensayo = integer(), corte = integer(),
                        Ganancia_cv_proyectada_millones = numeric(),
                        Ganancia_sd = numeric())

In [17]:
# Reviso el presupuesto y el espacio antes de ejecutar la búsqueda.
PARAM$hyperparametertuning

$xval_folds
[1] 5

$n_trials
[1] 80

$n_startup_trials
[1] 15

$num_iterations
[1] 1000

$espacio
$espacio$learning_rate
[1] 0.01 0.15

$espacio$num_leaves
[1]  16 128

$espacio$min_data_in_leaf
[1]   20 2000

$espacio$feature_fraction
[1] 0.5 1.0

$espacio$bagging_fraction
[1] 0.6 1.0


$espacio_efectivo
$espacio_efectivo$learning_rate
[1] 0.01 0.15

$espacio_efectivo$num_leaves
[1]  16 128

$espacio_efectivo$min_data_in_leaf
[1]   20 2000

$espacio_efectivo$feature_fraction
[1] 0.5 1.0

$espacio_efectivo$bagging_fraction
[1] 0.6 1.0

In [18]:
# ask/tell mantiene LightGBM en R. Optuna recibe GANANCIA, no AUC.
espacio <- PARAM$hyperparametertuning$espacio_efectivo
for (iteracion in seq_len(PARAM$hyperparametertuning$n_trials)) {
  trial <- estudio$ask()
  resultado <- tryCatch({
    x <- list(
      learning_rate = trial$suggest_float(
        "learning_rate", espacio$learning_rate[1],
        espacio$learning_rate[2], log = TRUE
      ),
      num_leaves = trial$suggest_int(
        "num_leaves", as.integer(espacio$num_leaves[1]),
        as.integer(espacio$num_leaves[2]), log = TRUE
      ),
      min_data_in_leaf = trial$suggest_int(
        "min_data_in_leaf", as.integer(espacio$min_data_in_leaf[1]),
        as.integer(espacio$min_data_in_leaf[2]), log = TRUE
      ),
      feature_fraction = trial$suggest_float(
        "feature_fraction", espacio$feature_fraction[1],
        espacio$feature_fraction[2]
      ),
      bagging_fraction = trial$suggest_float(
        "bagging_fraction", espacio$bagging_fraction[1],
        espacio$bagging_fraction[2]
      )
    )
    evaluacion <- Estimar_Ganancia_lightgbm(x)
    list(parametros = c(x, list(num_iterations = evaluacion$num_iterations)),
         evaluacion = evaluacion)
  }, error = function(e) {
    invisible(estudio$tell(trial, state = estado_fallido))
    stop(e)
  })

  evaluacion <- resultado$evaluacion
  trial$set_user_attr("num_iterations", as.integer(evaluacion$num_iterations))
  trial$set_user_attr("AUC", as.numeric(evaluacion$AUC))
  trial$set_user_attr("Ganancia_sd", as.numeric(evaluacion$Ganancia_sd))
  invisible(estudio$tell(trial, as.numeric(evaluacion$Ganancia)))

  # Número de ensayo 0-based, igual que Optuna; ensayo 0 es la referencia.
  tb_nueva <- rbindlist(list(tb_nueva, as.data.table(c(
    list(ensayo = as.integer(trial$number)), resultado$parametros,
    list(Ganancia_cv_proyectada_millones = evaluacion$Ganancia,
         Ganancia_sd = evaluacion$Ganancia_sd, AUC = evaluacion$AUC)
  ))), use.names = TRUE)
  curva <- copy(evaluacion$curva)
  curva[, ensayo := as.integer(trial$number)]
  setcolorder(curva, names(tb_curvas))
  tb_curvas <- rbindlist(list(tb_curvas, curva), use.names = TRUE)
  fwrite(tb_nueva, "tb_optuna_ganancia_1000.txt", sep = "\t")
  fwrite(tb_curvas, "tb_curvas_ganancia_1000.txt", sep = "\t")
  message("Ensayo ", iteracion, "/", PARAM$hyperparametertuning$n_trials,
          " | ganancia CV proyectada (millones): ", signif(evaluacion$Ganancia, 7),
          " | AUC secundaria: ", signif(evaluacion$AUC, 7),
          " | mejor ganancia: ", signif(estudio$best_value, 7))
}

Ensayo 1/80 | ganancia CV proyectada (millones): 580.724 | AUC secundaria: 0.9305595 | mejor ganancia: 580.724

Ensayo 2/80 | ganancia CV proyectada (millones): 547.6049 | AUC secundaria: 0.9173763 | mejor ganancia: 580.724

Ensayo 3/80 | ganancia CV proyectada (millones): 551.6187 | AUC secundaria: 0.9212209 | mejor ganancia: 580.724

Ensayo 4/80 | ganancia CV proyectada (millones): 588.7533 | AUC secundaria: 0.9299908 | mejor ganancia: 588.7533

Ensayo 5/80 | ganancia CV proyectada (millones): 521.5099 | AUC secundaria: 0.9068133 | mejor ganancia: 588.7533

Ensayo 6/80 | ganancia CV proyectada (millones): 555.6335 | AUC secundaria: 0.9204001 | mejor ganancia: 588.7533

Ensayo 7/80 | ganancia CV proyectada (millones): 552.6228 | AUC secundaria: 0.9209981 | mejor ganancia: 588.7533

Ensayo 8/80 | ganancia CV proyectada (millones): 587.7495 | AUC secundaria: 0.9308979 | mejor ganancia: 588.7533

Ensayo 9/80 | ganancia CV proyectada (millones): 556.6367 | AUC secundaria: 0.9192791 | mejo

In [19]:
# Veo los hiperparámetros y la AUC de los ensayos completados.
tb_nueva

ensayo,learning_rate,num_leaves,min_data_in_leaf,feature_fraction,bagging_fraction,num_iterations,Ganancia_cv_proyectada_millones,Ganancia_sd,AUC
<int>,<dbl>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
0,0.01000000,64,500,0.7500000,1.0000000,1000,580.7240,45.56157,0.9305595
1,0.13234126,61,217,0.7786096,0.9679285,1000,547.6049,25.72248,0.9173763
2,0.08126893,46,365,0.9504230,0.9149802,1000,551.6187,40.34385,0.9212209
3,0.01249456,128,24,0.5939015,0.9541372,1000,588.7533,31.23718,0.9299908
4,0.14900822,96,59,0.9824986,0.6582333,1000,521.5099,39.23947,0.9068133
5,0.08491510,62,21,0.5396869,0.8934352,1000,555.6335,39.01610,0.9204001
6,0.06836991,56,37,0.9159845,0.8152200,1000,552.6228,44.25218,0.9209981
7,0.01301550,58,44,0.5268722,0.6996484,1000,587.7495,41.60484,0.9308979
8,0.09365326,49,1136,0.7677841,0.8935703,1000,556.6367,43.35029,0.9192791


In [20]:
fwrite(tb_nueva, "tb_optuna_ganancia_1000.txt", sep = "\t")
fwrite(tb_curvas, "tb_curvas_ganancia_1000.txt", sep = "\t")

In [21]:
# Selecciono por ganancia y conservo la interfaz para producción.
setorder(tb_nueva, -Ganancia_cv_proyectada_millones, ensayo)
numero_ganador <- as.integer(estudio$best_trial$number)
ganador <- tb_nueva[ensayo == numero_ganador]
PARAM$out$lgbm$criterio_seleccion <- "ganancia_cv_proyectada_millones"
PARAM$out$lgbm$corte_objetivo <- PARAM$ganancia$envios_objetivo
PARAM$out$lgbm$Ganancia_cv_proyectada_millones <- as.numeric(estudio$best_value)
PARAM$out$lgbm$Ganancia_sd <- ganador$Ganancia_sd
PARAM$out$lgbm$AUC <- ganador$AUC
PARAM$out$lgbm$ganancia_referencia_5942 <-
  tb_nueva[ensayo == 0L, Ganancia_cv_proyectada_millones]
PARAM$out$lgbm$mejores_hiperparametros <- modifyList(
  estudio$best_params,
  list(num_iterations = as.integer(PARAM$hyperparametertuning$num_iterations))
)
# Comparación de todos los cortes para el ganador y la referencia.
tb_curvas[ensayo %in% c(0L, numero_ganador)][order(ensayo, corte)]

ensayo,corte,Ganancia_cv_proyectada_millones,Ganancia_sd
<int>,<int>,<dbl>,<dbl>
0,9000,570.8635,38.09054
0,9500,574.4515,40.65636
0,10000,582.9822,41.20731
0,10500,583.6094,40.99267
0,11000,582.1039,40.65290
0,11500,581.6022,40.02916
0,12000,581.2257,41.45037
0,12500,580.7240,45.56157
0,13000,579.2436,46.11565


In [22]:
write_yaml( PARAM, file="PARAM.yml")

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [23]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparametros

In [24]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [25]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [26]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [27]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 149911

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_freq
[1] 1

$bagging_fraction
[1] 0.8740107

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 1000

$learning_rate
[1] 0.01158069

$feature_fraction
[1] 0.5660003

$num_leaves
[1] 59

$min_data_in_leaf
[1] 64

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [28]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [29]:
  # entreno LightGBM

  modelo_final <- lgb.train(
    data= dtrain,
    param= param_normalizado
  )

In [30]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(modelo_final))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

In [31]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [32]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [33]:
# Verifico los IDs y la cantidad real de septiembre antes de exportar.
stopifnot(nrow(dfuture) == n_futuro,
          !anyNA(dfuture$numero_de_cliente), !anyDuplicated(dfuture$numero_de_cliente),
          length(prediccion) == n_futuro, all(is.finite(prediccion)))
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

Kaggle Competition Submit

In [34]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
  Sys.sleep(45)
}

In [35]:
write_yaml( PARAM, file="PARAM.yml")

In [36]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Oct 23 01:08:40 2026"

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-04**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-04**

Siéntase libre de agregar columnas a la hoja **TareaHogar-04**  en caso de ser necesario.